In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import datetime as dt

try:
    import snowflake.connector
except:
    ! pip install snowflake 
    import snowflake.connector

import snowflake.connector
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-01-23 00:03:44.177060


#### Functions

#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

str_date_min = '2024-11-26'

str_date_max = '2025-01-21'

Project: 20241112-simple-model-test
Task: parse_all_apps
Subtask: 01_pull_data


#### Connect

In [4]:
# load 
str_filename = 'datascience_rsa_key.p8'
str_local_path = f'./{str_filename}'
with open(str_local_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend(),
    )

# convert to bytes
private_key = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

# connect to snowflake
conn = snowflake.connector.connect(
    user='datascience', 
    private_key=private_key,
    account='pfs', 
    warehouse='datascience',
    database='raw',
    schema='source_s3_scorehistory',
)

#### Query

In [5]:
# query
str_query = f"""
WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '{str_date_min}' 
    AND DATE(request_datetime) < '{str_date_max}'
    AND response_model_name = 'PRESTIGE-GEN-XII'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;
"""
print(str_query)


WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '2024-11-26' 
    AND DATE(request_datetime) < '2025-01-21'
    AND response_model_name = 'PRESTIGE-GEN-XII'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;



#### Pull data

In [6]:
%%time

# pull payloads
df = pd.read_sql(
    sql=str_query,
    con=conn,
)
# show
df

<timed exec>:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 3min 18s, sys: 1min 42s, total: 5min 1s
Wall time: 5min 37s


,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
0,8544966/x3QS-2025-01-06-14:38:36,8544966,2025-01-06 21:38:36+00:00,"{\n ""request"": {\n ""request_id"": ""85449668...","{\n ""request_id"": ""8544966884627"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8544966884627,PRESTIGE-GEN-XII,V1,1
1,8506900/Z92q-2024-12-18-19:14:55,8506900,2024-12-19 02:14:55+00:00,"{\n ""request"": {\n ""request_id"": ""85069004...","{\n ""request_id"": ""8506900413352"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506900413352,PRESTIGE-GEN-XII,V1,1
2,8561362/4Ln9-2025-01-13-18:49:00,8561362,2025-01-14 01:49:00+00:00,"{\n ""request"": {\n ""request_id"": ""85613627...","{\n ""request_id"": ""85613627021"",\n ""rows"": [...","{\n ""Response"": [\n {\n ""CounterOffer...",85613627021,PRESTIGE-GEN-XII,V1,1
3,8474876/Nsi8-2024-12-09-21:37:38,8474876,2024-12-10 04:37:38+00:00,"{\n ""request"": {\n ""request_id"": ""84748766...","{\n ""request_id"": ""8474876689953"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8474876689953,PRESTIGE-GEN-XII,V1,1
4,8465170/c69Z-2024-12-05-17:24:35,8465170,2024-12-06 00:24:35+00:00,"{\n ""request"": {\n ""request_id"": ""84651705...","{\n ""request_id"": ""8465170579834"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8465170579834,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
88657,8469056/3lgV-2024-12-06-21:30:33,8469056,2024-12-07 04:30:33+00:00,"{\n ""request"": {\n ""request_id"": ""84690563...","{\n ""request_id"": ""8469056369541"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8469056369541,PRESTIGE-GEN-XII,V1,1
88658,8560281/i571-2025-01-11-22:30:42,8560281,2025-01-12 05:30:42+00:00,"{\n ""request"": {\n ""request_id"": ""85602816...","{\n ""request_id"": ""8560281603978"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8560281603978,PRESTIGE-GEN-XII,V1,1
88659,8506082/Gy54-2024-12-18-15:59:13,8506082,2024-12-18 22:59:13+00:00,"{\n ""request"": {\n ""request_id"": ""85060823...","{\n ""request_id"": ""8506082398019"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506082398019,PRESTIGE-GEN-XII,V1,1
88660,8554968/8FHE-2025-01-09-23:57:37,8554968,2025-01-10 06:57:37+00:00,"{\n ""request"": {\n ""request_id"": ""85549685...","{\n ""request_id"": ""8554968590809"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8554968590809,PRESTIGE-GEN-XII,V1,1


#### Query for DLv1 accounts

In [7]:
str_query = f"""
SELECT ACCOUNTID
FROM 
    raw.source_s3_scorehistory.PAYLOAD_PARSED
WHERE DATE(request_datetime) >= '{str_date_min}' 
AND DATE(request_datetime) < '{str_date_max}'
AND response_model_name = 'PRESTIGE-DLV1'
"""

#### Pull data

In [8]:
%%time

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# get list of accounts
list_accountid = list(df_tmp['ACCOUNTID'])
# save memory
del df_tmp

<timed exec>:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 50.1 ms, sys: 825 μs, total: 50.9 ms
Wall time: 3.96 s


#### Remove DLv1 accounts

In [9]:
df = df[~df['ACCOUNTID'].isin(list_accountid)].copy()
# show
df

,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
0,8544966/x3QS-2025-01-06-14:38:36,8544966,2025-01-06 21:38:36+00:00,"{\n ""request"": {\n ""request_id"": ""85449668...","{\n ""request_id"": ""8544966884627"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8544966884627,PRESTIGE-GEN-XII,V1,1
1,8506900/Z92q-2024-12-18-19:14:55,8506900,2024-12-19 02:14:55+00:00,"{\n ""request"": {\n ""request_id"": ""85069004...","{\n ""request_id"": ""8506900413352"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506900413352,PRESTIGE-GEN-XII,V1,1
2,8561362/4Ln9-2025-01-13-18:49:00,8561362,2025-01-14 01:49:00+00:00,"{\n ""request"": {\n ""request_id"": ""85613627...","{\n ""request_id"": ""85613627021"",\n ""rows"": [...","{\n ""Response"": [\n {\n ""CounterOffer...",85613627021,PRESTIGE-GEN-XII,V1,1
3,8474876/Nsi8-2024-12-09-21:37:38,8474876,2024-12-10 04:37:38+00:00,"{\n ""request"": {\n ""request_id"": ""84748766...","{\n ""request_id"": ""8474876689953"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8474876689953,PRESTIGE-GEN-XII,V1,1
4,8465170/c69Z-2024-12-05-17:24:35,8465170,2024-12-06 00:24:35+00:00,"{\n ""request"": {\n ""request_id"": ""84651705...","{\n ""request_id"": ""8465170579834"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8465170579834,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
88657,8469056/3lgV-2024-12-06-21:30:33,8469056,2024-12-07 04:30:33+00:00,"{\n ""request"": {\n ""request_id"": ""84690563...","{\n ""request_id"": ""8469056369541"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8469056369541,PRESTIGE-GEN-XII,V1,1
88658,8560281/i571-2025-01-11-22:30:42,8560281,2025-01-12 05:30:42+00:00,"{\n ""request"": {\n ""request_id"": ""85602816...","{\n ""request_id"": ""8560281603978"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8560281603978,PRESTIGE-GEN-XII,V1,1
88659,8506082/Gy54-2024-12-18-15:59:13,8506082,2024-12-18 22:59:13+00:00,"{\n ""request"": {\n ""request_id"": ""85060823...","{\n ""request_id"": ""8506082398019"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506082398019,PRESTIGE-GEN-XII,V1,1
88660,8554968/8FHE-2025-01-09-23:57:37,8554968,2025-01-10 06:57:37+00:00,"{\n ""request"": {\n ""request_id"": ""85549685...","{\n ""request_id"": ""8554968590809"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8554968590809,PRESTIGE-GEN-XII,V1,1


#### Close connection

In [10]:
conn.close()

#### Write to s3

In [11]:
%%time

str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:283: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 11min 53s, sys: 1min 42s, total: 13min 35s
Wall time: 15min 21s
